<a href="https://colab.research.google.com/github/jameshphan-png/Coding-Exercise---ML-Basics/blob/dev/Module_3_Coding_Exercise_Part_3_(Customer_Segmentation).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# ------------------------------------------------------------
# IMPORTANT (READ ME BEFORE STARTING)
#
# Ensure you have downloaded "Mall_Customers.csv" as the excel file obtained from Github.
# USE THE EXCEL FILE AND UPLOAD INTO THE RUNTIME IN COLAB THROUGH THE 'Files' (ON LEFT SIDE BAR), AND DRAG AND DROP THERE.
# ------------------------------------------------------------

# ======================================================
# Dataset Source:
# Kaggle - "Mall Customer Segmentation" Dataset
# https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python
# ======================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score


# -----------------------------
# 1) Config Cluster Values for K means
# -----------------------------

#Setting constraints to K (cluster value)
# For the sake of configuration, we will refer it in between 2 to 11 as the constraints as setting K_Min value to 1 will make it undefineable for clusters (counts as 0)
#Instead, K_Min will be based on a value of 2, having 1 cluster to compare, all the way up to K_Max which is a value based on 11, having 10 clusters to compare as the max.
K_MIN = 2
K_MAX = 11
RANDOM_STATE = 42    #Seed generation

# If you want to force a K instead of auto-selection, set to an int (e.g., 5); else None
FORCE_K = None


# -----------------------------
# 2) Load Data
# -----------------------------

#Reads the Mall_Customers.csv Excel file
DATA_PATH = "Mall_Customers.csv"
df = pd.read_csv(DATA_PATH)

# Remove ID columns if present (for data cleanup on null values)
id_like_cols = [c for c in df.columns if c.strip().lower() in {"customerid", "id"}]
df_model = df.drop(columns=id_like_cols, errors="ignore").copy()


# -----------------------------
# 3) Build feature matrix - Create a structure table for future reference
#    - numeric scaled
#    - categoricals one-hot encoded
# -----------------------------
numeric_cols = df_model.select_dtypes(include=[np.number]).columns.tolist() #Based on numerical columns in the dataset
categorical_cols = [c for c in df_model.columns if c not in numeric_cols] #Based on categorical "name" columns in the dataset

# OneHotEncode categoricals (Converting categorical "name" columns for the feature matrix table structure)
df_cat = pd.get_dummies(df_model[categorical_cols], drop_first=True) if categorical_cols else pd.DataFrame(index=df_model.index)

# Scale numeric values (IMPORTANT for K-Means distance fairness)
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(df_model[numeric_cols])

# Creates matrix based on clustering
X_scaled = np.hstack([X_num_scaled, df_cat.values]) if not df_cat.empty else X_num_scaled

#Explains basic features for this dataset, including the matrix shape.
print("\n=== FEATURES USED ===")
print("Numeric (scaled):", numeric_cols)
print("Categorical (one-hot):", categorical_cols if categorical_cols else "None")
print("\nFeature matrix shape:", X_scaled.shape)
print("> Matrix shape (rows, columns) means: (#customers, #features used for clustering).")
print("> Example: (200, 4) = 200 customers described by 4 features.\n")


# -----------------------------
# 4) Elbow + Silhouette for K selection
# -----------------------------

#Discovers & researches the number of clusters, finding the K value that is the most optimal (ranging from 1 to 10)
ks = list(range(1, K_MAX))  # 1..10
inertias = []
silhouettes = [np.nan]      # silhouette undefined for k=1

#Value testing to see which K value is ideal, before creating the plots (for both Elbow & Silhouette)
for k in ks:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init="auto")
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    if k >= 2:
        silhouettes.append(silhouette_score(X_scaled, labels))

# Decision table for justification
#This part contributes  the "executive summary" for the cluster profile segmentations, which will be printed as an excel file later on."
metrics = pd.DataFrame({
    "K": ks,
    "Inertia": inertias,
    "Silhouette": silhouettes
})

# Justifies the reason why the specific "K" value is chosen:
candidate = metrics[metrics["K"].between(K_MIN, K_MAX - 1)].copy()

# Inertia drop trend (helps identify elbow-ish zone)
candidate["Inertia_Drop"] = candidate["Inertia"].shift(1) - candidate["Inertia"]
candidate["Inertia_Drop"] = candidate["Inertia_Drop"].fillna(candidate["Inertia_Drop"].median())

# Elbow-ish zone: keep Ks where improvement is still meaningful
elbow_zone = candidate[candidate["Inertia_Drop"] >= candidate["Inertia_Drop"].median()]
search_space = elbow_zone if len(elbow_zone) else candidate

auto_k = int(search_space.loc[search_space["Silhouette"].idxmax(), "K"])
optimal_k = int(FORCE_K) if isinstance(FORCE_K, int) else auto_k

# Save elbow plot (inertia) - Creates the Elbow Plot (shows which K value is the most optimal to create)
plt.figure(figsize=(9, 5))
plt.plot(ks, inertias, marker="o")
plt.xticks(ks)
plt.xlabel("K (number of clusters)")
plt.ylabel("Inertia (SSE) - lower is better")
plt.title("Elbow Plot (Inertia vs K)")
plt.tight_layout()
plt.savefig("elbow_plot.png", dpi=150)
plt.close()

# Save silhouette plot (separation quality) - Creates the silhouette plot (the K value that is the most optimal at best in reference to Elbow Plot)
plt.figure(figsize=(9, 5))
plt.plot(ks, silhouettes, marker="o")
plt.xticks(ks)
plt.xlabel("K (number of clusters)")
plt.ylabel("Silhouette Score - higher is better")
plt.title("Silhouette Plot (Silhouette vs K)")
plt.tight_layout()
plt.savefig("silhouette_plot.png", dpi=150)
plt.close()

#This segment models a table on why the optimal value for k cluster is chosen & evidence to understand why that value is chosen
print("=== K SELECTION EVIDENCE (for justification) ===")
print(metrics.to_string(index=False, justify="center", formatters={
    "Inertia": "{:.2f}".format,
    "Silhouette": (lambda x: "  -" if pd.isna(x) else f"{x:.4f}")
}))
print(f"\nSelected K = {optimal_k}")

# Plain-English justification (identifies the values for best silhouette, and inertia for future explanations)
best_sil = metrics.loc[metrics["K"] == optimal_k, "Silhouette"].values[0]
inertia_k = metrics.loc[metrics["K"] == optimal_k, "Inertia"].values[0]
inertia_prev = metrics.loc[metrics["K"] == max(1, optimal_k - 1), "Inertia"].values[0]

#The improve value defines the "improvement" compact based on the value that begins to diminish significantly less
improve = inertia_prev - inertia_k

print("\nJustification:")
print("- Data was standardized (mean=0, std=1) so Age/Income/Spending contribute fairly in distance calculations.")
print("- Elbow plot ('elbow_plot.png') helps identify where inertia improvements start to slow down.")
print("- Silhouette plot ('silhouette_plot.png') checks how well-separated the clusters are.")
print(f"- K={optimal_k} is selected because it has solid separation (silhouette={best_sil:.4f}) "
      f"and still improves compactness (Δ inertia vs K-1 ≈ {improve:.2f}).")
print("- REFER TO THE 'EXTRA FILES FOR REFERENCES' FOR MORE INFORMATION")


# -----------------------------
# 5) Fit final model (1-based cluster labels)
# -----------------------------

#Assigns data points into their specified groups
kmeans = KMeans(n_clusters=optimal_k, random_state=RANDOM_STATE, n_init="auto")
labels_0_based = kmeans.fit_predict(X_scaled)

# Convert to 1-based labels for readability (Cluster 1..K instead of 0..K-1)
df["cluster"] = labels_0_based + 1 #So that clusters do not start at 0, but at 1 instead.

#Explains the interpretation of clusters & extra information given
print("\n=== CLUSTERING INTERPRETATION ===")
print(f"K = {optimal_k} means the algorithm will create and use {optimal_k} distinct clusters (segments).")
print("A 'cluster' is a group of customers who are similar to each other based on Age, Income, and Spending behavior.")
print("Clusters help because they simplify large customer data into a few clear groups so marketing can be targeted.")
print("Clusters are labeled 1..K for reporting clarity.")
print("Note: Variable 'n' means the number of customers in that cluster.\n")


# ----------------------------------------------------------
# 6) Create Summaries using 1-based clusters
# ----------------------------------------------------------

#Calculates the averages for the numerical columns
cluster_means = df.groupby("cluster")[numeric_cols].mean()

#Clusters are sorted & given their respective sizes
cluster_sizes = df["cluster"].value_counts().sort_index()


# ----------------------------------------------------------
# 7) Detect column names, and sort it through listing them individually
# ----------------------------------------------------------

#Locates specific column frames and adjusts it for lowercased characters for convenience
def pick_col_contains(substr_list):
    for c in df.columns:
        cl = c.strip().lower()
        if any(s in cl for s in substr_list):
            return c
    return None

#Groups/Formats columns based on income, spending, and age, which will be used as a factor for cluster grouping later on
income_col = pick_col_contains(["annual income", "income"])
spend_col  = pick_col_contains(["spending score", "spending"])
age_col    = pick_col_contains(["age"])

#Lists the following columns for reference to produce the data results
print("=== COLUMN LISTING (REFERENCE) ===")
print("The following columns will be used to list out & analyze the clusters in detail:")
print("Income column:", income_col, "- NOTE: k$ is IN THOUSANDS")
print("Spending column:", spend_col)
print("Age column:", age_col)


# ---------------------------------------------------------------------------------------
# 8) Cluster explanations & realistic marketing strategies (print Cluster 1..K)
# ---------------------------------------------------------------------------------------
print("\n=== CLUSTER EXPLANATIONS & REALISTIC MARKETING STRATEGIES ===")

#All cluster rows listed in a list (will be formatted later on)
cluster_rows = []

#This section defines the quantiles, which will be sorted based on Low, Mid, High, assigning data rows into their respective category
def quantiles(col):
    # 33% and 67% split => Low / Mid / High buckets
    return df[col].quantile(0.33), df[col].quantile(0.67)

#Quantile sorting based on income, spending, and age categories
income_q = quantiles(income_col)
spend_q  = quantiles(spend_col)
age_q    = quantiles(age_col)

#This section filters clusters based on a certain threshold system (given the base factors of income/spending/age)
#All 3 factors will be weighted down on an average metric trained by the dataset (Mall_Customers.csv values) to determine others and identify the designated quantile
def band(value, low_thr, high_thr):
    if value <= low_thr:
        return "Low"
    elif value >= high_thr:
        return "High"
    return "Mid"

for cl in range(1, optimal_k + 1):
    n = int(cluster_sizes.loc[cl])
    row = {"cluster": cl, "size": n}

    inc = cluster_means.loc[cl, income_col]
    sp  = cluster_means.loc[cl, spend_col]
    ag  = cluster_means.loc[cl, age_col]

    inc_band = band(inc, income_q[0], income_q[1])
    sp_band  = band(sp, spend_q[0], spend_q[1])
    ag_band  = band(ag, age_q[0], age_q[1])

    label = f"{inc_band} Income / {sp_band} Spending / {ag_band} Age"

    # Realistic mall/retail strategies:
    # (These are more detailed and tied to real marketing tactics.

    #Based on grouping, depending on which segment one person belongs to or characteristics that match from income, spending, or gender
    #Covers the "Highs" and "Lows" segments
    if inc_band == "High" and sp_band == "High":
        segment_explain = (
            "VIP / Premium Loyalists: customers with strong purchasing power and high willingness to spend."
        )
        strategy = (
            "Marketing Strategy (VIP / Premium Loyalists):\n"
            "- Create VIP loyalty tiers (exclusive points multipliers, birthday perks, early access).\n"
            "- Offer premium bundles and curated collections (increase average order value).\n"
            "- Invite-only events (member shopping nights, brand previews).\n"
            "- Concierge-style outreach (personal shopper email/SMS for top customers).\n"
            "- Retention focus: avoid heavy discounting; prioritize exclusivity and service."
        )

    elif inc_band == "High" and sp_band == "Low":
        segment_explain = (
            "Affluent but Under-engaged: they can spend, but currently show low spending interest."
        )
        strategy = (
            "Marketing Strategy (High Potential / Under-engaged):\n"
            "- Reduce friction: free returns, strong product guarantees, clear quality messaging.\n"
            "- Personalized recommendations (based on browsing/category preferences).\n"
            "- Offer 'premium trial' perks (free alterations, free gift, or free delivery).\n"
            "- Invite to experiences (product demos, private previews) to build emotional engagement.\n"
            "- Goal: convert them into consistent spenders without relying on big discounts."
        )

    elif inc_band == "Low" and sp_band == "High":
        segment_explain = (
            "Deal-driven Enthusiasts: lower income but high spending score—often motivated by value and promotions."
        )
        strategy = (
            "Marketing Strategy (Deal-driven Enthusiasts):\n"
            "- Use flash sales and limited-time offers (urgency works well).\n"
            "- Offer bundles (buy 2 get 1, combo deals) to protect margins.\n"
            "- Promote 'Best under $X' collections.\n"
            "- Use points multipliers instead of big discounts (cheaper for the business).\n"
            "- Retargeting ads for sale items and price drops (high conversion likelihood)."
        )

    elif inc_band == "Low" and sp_band == "Low":
        segment_explain = (
            "Low Engagement / Low Value: lower ability to spend and low interest—often inactive or occasional shoppers."
        )
        strategy = (
            "Marketing Strategy (Low Engagement / Win-back):\n"
            "- Reactivation campaigns: welcome-back coupon, cart reminders, seasonal re-entry offers.\n"
            "- Entry-level bundles (starter packs) and low-cost add-ons at checkout.\n"
            "- Simplify messaging: fewer choices, clear value proposition.\n"
            "- Test channels: SMS vs email vs retargeting to find the cheapest conversion path.\n"
            "- Goal: increase visit frequency and first/second purchase conversion."
        )

    else:
        # Covers Mid/Mid segments and mixed segments not captured above for the unidentified segments match (whether high or low)
        segment_explain = (
            "Mainstream / Steady Shoppers: moderate income and/or spending—likely your largest stable segment."
        )
        strategy = (
            "Marketing Strategy (Mainstream / Growth):\n"
            "- Cross-sell & upsell: recommend complementary items at checkout.\n"
            "- Seasonal campaigns and personalized email flows (new arrivals, holiday sets).\n"
            "- A/B test incentives (free shipping vs % off) to optimize conversions.\n"
            "- Encourage repeat purchases: reminder campaigns based on typical repurchase windows.\n"
            "- Goal: gradually increase spending and frequency through personalization."
        )

    # Stores summary row (in part to finish creating the executive view excel file)
    row.update({
        "segment_label": label,
        "segment_explanation": segment_explain,
        "avg_income": round(float(inc), 2),
        "avg_spend": round(float(sp), 2),
        "avg_age": round(float(ag), 2),
        "strategy": strategy
    })
    cluster_rows.append(row)

    # Print explanation
    #This part explains the basic features of a segment individually, given its unique values & interpreted by Machine Learning in the original dataset
    #This section of the output is produced 6 times.
    print(f"\n--- Cluster {cl} (n={n}) ---")
    print("Segment label:", label)
    print("What this cluster likely represents:", segment_explain)
    print(f"Avg {income_col}: {inc:.2f} (k$)")
    print(f"Avg {spend_col}: {sp:.2f} (1-100)")
    print(f"Avg {age_col}: {ag:.2f}")

    #Explains Marketing Actions that may potentially work in favor to the segmented group of customers uniquely, based on income, age, etc.
    print("\nMarketing actions:")
    print(strategy)


# -----------------------------
# 9) Save outputs
# -----------------------------
#This creates the excel file for "executive view", showing insights based on all mentioned clusters.
cluster_profile = pd.DataFrame(cluster_rows)
cluster_profile.to_csv("cluster_profile_executive.csv", index=False)

df.to_csv("customer_segments_kmeans.csv", index=False)

#Extra files to refer to here:
print("\n=== EXTRA FILES FOR REFERENCES ===")
print("- elbow_plot.png - Shows diminishing inertia improvements after the elbow region")
print("- silhouette_plot.png - Supports separation quality")
print("- customer_segments_kmeans.csv - All customers with their assigned cluster label (1..K)")
print("- cluster_profile_executive.csv - Summary of clusters + averages + detailed strategies")



=== FEATURES USED ===
Numeric (scaled): ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
Categorical (one-hot): ['Gender']

Feature matrix shape: (200, 4)
> Matrix shape (rows, columns) means: (#customers, #features used for clustering).
> Example: (200, 4) = 200 customers described by 4 features.

=== K SELECTION EVIDENCE (for justification) ===
 K Inertia Silhouette
 1  649.28      NaN  
 2  438.52   0.3032  
 3  345.21   0.3120  
 4  254.36   0.3504  
 5  216.78   0.3498  
 6  181.95   0.3565  
 7  171.37   0.3316  
 8  153.30   0.3362  
 9  142.72   0.3118  
10  133.33   0.3087  

Selected K = 6

Justification:
- Data was standardized (mean=0, std=1) so Age/Income/Spending contribute fairly in distance calculations.
- Elbow plot ('elbow_plot.png') helps identify where inertia improvements start to slow down.
- Silhouette plot ('silhouette_plot.png') checks how well-separated the clusters are.
- K=6 is selected because it has solid separation (silhouette=0.3565) and still im